<a href="https://colab.research.google.com/github/marina-popova11/MediaImpactOnCryptoPrices/blob/training/model_on_three_classes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from datasets import DatasetDict, Dataset, load_from_disk
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoModel,
    AutoConfig,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)

import wandb
wandb.init(mode="offline")

def load_arrow(filepath):
    try:
        dataset = load_from_disk(filepath)
        print("Successfully loaded dataset from disk")
        return dataset
    except Exception as e:
        print(e)
        return None

dataset_path = "/content/drive/MyDrive/uniform_distribution_thresh_002"

import os
if os.path.exists(dataset_path):
    print(f"Path exists: {os.path.exists(dataset_path)}")
    print(f"Files in directory: {os.listdir(dataset_path)}")
else:
    print(f"Path does not exist. Current directory: {os.getcwd()}")
    print(f"Available files: {os.listdir('.')}")

dataset = load_arrow(dataset_path)

if dataset is None:
    try:
        dataset = DatasetDict({
            'train': load_from_disk(f"{dataset_path}/train"),
            'test': load_from_disk(f"{dataset_path}/test"),
            'validation': load_from_disk(f"{dataset_path}/validation")
        })
        print("dataset download throw DatasetDict")
    except Exception as e:
        print(e)
        exit()

print(f"\nDataset type: {type(dataset)}")
print(f"Dataset keys: {list(dataset.keys())}")

text_column = "text"
label_column = "label_class_6"

label_map = {-1: 0, 0: 1, 1: 2}
def encode_labels(batch):
    batch["label"] = [label_map[int(x)] for x in batch[label_column]]
    return batch

dataset = dataset.map(encode_labels, batched=True)
dataset = dataset.remove_columns([label_column])
print(dataset["train"]["label"][:10])
print(set(dataset["train"]["label"]))

model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    return tokenizer(
        examples[text_column],
        padding='max_length',
        truncation=True,
        max_length=128
    )

tokenized_datasets = DatasetDict()
for name, data in dataset.items():
    tokenized_datasets[name] = data.map(
        tokenize_function,
        batched=True,
        batch_size=1000
    )

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)


Path exists: True
Files in directory: ['dataset_dict.json', 'train', 'test', 'validation']
Successfully loaded dataset from disk

Dataset type: <class 'datasets.dataset_dict.DatasetDict'>
Dataset keys: ['train', 'test', 'validation']


Map:   0%|          | 0/21761 [00:00<?, ? examples/s]

Map:   0%|          | 0/3477 [00:00<?, ? examples/s]

Map:   0%|          | 0/4121 [00:00<?, ? examples/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/21761 [00:00<?, ? examples/s]

Map:   0%|          | 0/3477 [00:00<?, ? examples/s]

Map:   0%|          | 0/4121 [00:00<?, ? examples/s]

In [3]:
columns_to_keep = ["input_ids", "attention_mask", "label"]
for name in tokenized_datasets.keys():
    tokenized_datasets[name].set_format(
        type="torch",
        columns=columns_to_keep
    )

print(tokenized_datasets["train"]["label"][:5])
print(type(tokenized_datasets["train"]["label"][0]))

tensor([0, 0, 0, 0, 0])
<class 'torch.Tensor'>


In [7]:
import torch.nn as nn
import torch.nn.functional as F
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

labels = np.array(tokenized_datasets["train"]["label"])
class_counts = np.bincount(labels)
class_weights = 1.0 / class_counts
class_weights = class_weights / class_weights.sum()

class_weights = torch.tensor(class_weights, dtype=torch.float)
print("Class weights:", class_weights)

class WeightedTrainer(Trainer):
    def __init__(self, class_weights=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits

        loss_fct = nn.CrossEntropyLoss(weight=self.class_weights.to(logits.device))
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

Class weights: tensor([0.3215, 0.3766, 0.3019])


In [8]:
from transformers import DistilBertForSequenceClassification

model = DistilBertForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3
)

training_args = TrainingArguments(
    output_dir="./bert_price_direction",
    overwrite_output_dir=True,
    learning_rate=5e-6,
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    warmup_ratio=0.1,
    weight_decay=0.05,
    lr_scheduler_type="linear",
    logging_dir="./logs",
    logging_steps=100,
    eval_strategy="steps",
    eval_steps=300,
    save_steps=600,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    report_to=None,
    save_total_limit=5
)

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    pred_counts = np.bincount(predictions, minlength=3)
    print(f"Predicted class counts: {pred_counts}")
    return {
        'accuracy': accuracy_score(labels, predictions),
        "f1_macro": f1_score(labels, predictions, average="macro"),
        "f1_weighted": f1_score(labels, predictions, average="weighted"),
    }

trainer = WeightedTrainer(
    model=model,
    class_weights=class_weights,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=5)]
)

train_result = trainer.train()
trainer.save_model("./fine_tuned_distilbert_price_direction")
tokenizer.save_pretrained("./fine_tuned_distilbert_price_direction")

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipython-input-3387424174.py:16: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `WeightedTrainer.__init__`. Use `processing_class` instead.
  super().__init__(*args, **kwargs)


Step,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 Weighted
300,1.098400,1.089744,0.396506,0.320157,0.363050
600,1.097500,1.097239,0.361077,0.298268,0.319956
900,1.092500,1.088438,0.386557,0.351996,0.380134
1200,1.089600,1.088335,0.385586,0.356262,0.382269
1500,1.085400,1.087281,0.385586,0.343702,0.374683
1800,1.085900,1.090894,0.387527,0.368031,0.388768


Predicted class counts: [ 199 2235 1687]
Predicted class counts: [ 219  964 2938]
Predicted class counts: [ 975 2099 1047]
Predicted class counts: [ 749 1878 1494]
Predicted class counts: [ 503 1907 1711]
Predicted class counts: [1029 1799 1293]


('./fine_tuned_distilbert_price_direction/tokenizer_config.json',
 './fine_tuned_distilbert_price_direction/special_tokens_map.json',
 './fine_tuned_distilbert_price_direction/vocab.txt',
 './fine_tuned_distilbert_price_direction/added_tokens.json',
 './fine_tuned_distilbert_price_direction/tokenizer.json')

In [9]:
preds = trainer.predict(tokenized_datasets["validation"])
y_pred = np.argmax(preds.predictions, axis=1)
y_true = preds.label_ids

from sklearn.metrics import classification_report, confusion_matrix
print(classification_report(y_true, y_pred, target_names=["down (-1)", "neutral (0)", "up (1)"]))
cm = confusion_matrix(y_true, y_pred)
print(cm)

Predicted class counts: [1029 1799 1293]
              precision    recall  f1-score   support

   down (-1)       0.30      0.30      0.30      1011
 neutral (0)       0.49      0.47      0.48      1862
      up (1)       0.32      0.33      0.33      1248

    accuracy                           0.39      4121
   macro avg       0.37      0.37      0.37      4121
weighted avg       0.39      0.39      0.39      4121

[[304 393 314]
 [418 879 565]
 [307 527 414]]


In [10]:
test_predictions = trainer.predict(tokenized_datasets["test"])
y_true = test_predictions.label_ids
y_pred = np.argmax(test_predictions.predictions, axis=1)

print(classification_report(
    y_true,
    y_pred,
    target_names=["down", "neutral", "up"],
    digits=3
))

cm = confusion_matrix(y_true, y_pred)
print("Confusion matrix:\n", cm)

cm_norm = cm / cm.sum(axis=1, keepdims=True)
print("Normalized confusion matrix:\n", cm_norm)

Predicted class counts: [1146  692 1639]
              precision    recall  f1-score   support

        down      0.298     0.338     0.317      1012
     neutral      0.364     0.201     0.259      1253
          up      0.351     0.475     0.404      1212

    accuracy                          0.336      3477
   macro avg      0.338     0.338     0.327      3477
weighted avg      0.341     0.336     0.326      3477

Confusion matrix:
 [[342 200 470]
 [408 252 593]
 [396 240 576]]
Normalized confusion matrix:
 [[0.33794466 0.19762846 0.46442688]
 [0.32561852 0.20111732 0.47326417]
 [0.32673267 0.1980198  0.47524752]]
